# Vinos: estructura, agrupamiento y pérdida de información

**Carlos Mario Salvatore Ocampo Aguilar · Introducción a Machine Learning**

Este cuaderno utiliza las 13 mediciones químicas del conjunto Wine. Se estudian K-means, Ward y una mezcla gaussiana, junto con PCA y SVD. Las clases de procedencia se conservan únicamente para contrastar las particiones al final: no son entradas de los algoritmos.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

OUT = Path('resultados')
OUT.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10, 'axes.spines.top': False,
                     'axes.spines.right': False})

from sklearn.datasets import load_wine
from sklearn.mixture import GaussianMixture

In [2]:
datos = load_wine()
X_raw, y = datos.data, datos.target
X = StandardScaler().fit_transform(X_raw)
print(f'Matriz: {X.shape}; faltantes: {np.isnan(X_raw).sum()}')
print('Clases de referencia (no usadas para agrupar):', np.bincount(y))
print('Rango de desviaciones originales:', X_raw.std(axis=0).min().round(2),
      'a', X_raw.std(axis=0).max().round(2))

Matriz: (178, 13); faltantes: 0
Clases de referencia (no usadas para agrupar): [59 71 48]
Rango de desviaciones originales: 0.12 a 314.02


## 1. Escala y elección de k

La estandarización impide que una variable domine por su unidad de medida. Se examinan k=2,…,6 con inercia y silueta. La guía pide explicar el codo, pero no se afirma un mínimo matemático donde solo hay una curva descendente. Se compara k=3 para evaluar tres grupos interpretables.

In [3]:
rows = []
for k in range(2, 7):
    fit = KMeans(n_clusters=k, n_init=20, random_state=42).fit(X)
    rows.append({'k': k, 'inercia': fit.inertia_,
                 'silueta': silhouette_score(X, fit.labels_)})
sweep = pd.DataFrame(rows)
display(sweep.round(4))
sweep.to_csv(OUT / 'seleccion_k.csv', index=False)

fig, axs = plt.subplots(1, 2, figsize=(9, 3.3))
axs[0].plot(sweep.k, sweep.inercia, 'o-', color='#294c60')
axs[0].set(xlabel='Número de grupos k', ylabel='Inercia', title='Compactación')
axs[1].plot(sweep.k, sweep.silueta, 'o-', color='#bd6b38')
axs[1].set(xlabel='Número de grupos k', ylabel='Silueta media', title='Separación')
for ax in axs: ax.set_xticks(sweep.k)
fig.tight_layout()
fig.savefig(OUT / 'seleccion_k.png', bbox_inches='tight')
plt.show()

,k,inercia,silueta
0,2,1658.7589,0.2593
1,3,1277.9285,0.2849
2,4,1175.3519,0.2586
3,5,1107.0073,0.2315
4,6,1046.0023,0.2372


C:\Users\karlo\AppData\Local\Temp\ipykernel_29204\1040676689.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Algoritmos con supuestos distintos

K-means crea regiones alrededor de centroides; Ward construye una jerarquía; la mezcla gaussiana admite grupos con covarianzas distintas. Los tres reciben las mismas variables estandarizadas. Davies–Bouldin y silueta evalúan separación interna; ARI contrasta con las clases conocidas sin haberlas usado para ajustar.

In [4]:
km = KMeans(n_clusters=3, n_init=20, random_state=42).fit(X)
ward = AgglomerativeClustering(n_clusters=3, linkage='ward').fit(X)
gmm = GaussianMixture(n_components=3, covariance_type='full',
                      n_init=10, random_state=42).fit(X)
pred = {'K-means': km.labels_, 'Ward': ward.labels_,
        'Mezcla gaussiana': gmm.predict(X)}
comparacion = pd.DataFrame([
    {'método': name, 'silueta': silhouette_score(X, lab),
     'Davies-Bouldin': davies_bouldin_score(X, lab),
     'ARI externo': adjusted_rand_score(y, lab),
     'tamaños': '/'.join(map(str, np.bincount(lab, minlength=3)))}
    for name, lab in pred.items()])
display(comparacion.round(4))
comparacion.to_csv(OUT / 'comparacion_metodos.csv', index=False)
print('ARI K-means frente a Ward:', round(adjusted_rand_score(km.labels_, ward.labels_), 4))

,método,silueta,Davies-Bouldin,ARI externo,tamaños
0,K-means,0.2849,1.3892,0.8975,65/51/62
1,Ward,0.2774,1.4186,0.7899,58/56/64
2,Mezcla gaussiana,0.2844,1.3938,0.8804,63/64/51


ARI K-means frente a Ward: 0.853


## 3. Reducir trece mediciones a dos ejes

PCA resume varianza tras centrar los datos; SVD de esa matriz centrada produce el mismo subespacio. La reconstrucción de rango 2 mide el costo de reducir. También se muestran las cargas para ver qué variables explican la proyección, sin tratarla como una clasificación.

In [5]:
pca = PCA(n_components=2).fit(X)
Z = pca.transform(X)
U, singular, Vt = np.linalg.svd(X - X.mean(axis=0), full_matrices=False)
X2 = (U[:, :2] * singular[:2]) @ Vt[:2, :] + X.mean(axis=0)
X2_pca = pca.inverse_transform(Z)
error_relativo = np.linalg.norm(X - X2) / np.linalg.norm(X - X.mean(axis=0))
coincidencia = np.max(np.abs(X2 - X2_pca))
varianza_2d = float(pca.explained_variance_ratio_.sum())
print(f'Varianza explicada por 2 CP: {varianza_2d:.4f}')
print(f'Error relativo de reconstrucción SVD rango 2: {error_relativo:.4f}')
print(f'Diferencia máxima entre reconstrucciones PCA y SVD: {coincidencia:.2e}')

Varianza explicada por 2 CP: 0.5541
Error relativo de reconstrucción SVD rango 2: 0.6678
Diferencia máxima entre reconstrucciones PCA y SVD: 3.33e-15


In [6]:
fig, axs = plt.subplots(1, 2, figsize=(9, 3.6), sharex=True, sharey=True)
for ax, (name, labels) in zip(axs, list(pred.items())[:2]):
    ax.scatter(Z[:, 0], Z[:, 1], c=labels, cmap='cividis', s=28,
               alpha=.85, edgecolors='white', linewidths=.2)
    ax.set(xlabel='CP1', ylabel='CP2', title=name)
fig.suptitle('Dos particiones sobre la proyección PCA')
fig.tight_layout()
fig.savefig(OUT / 'grupos_pca.png', bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(pca.components_, cmap='RdBu_r', vmin=-.55, vmax=.55, aspect='auto')
etiquetas = ['Alcohol', 'Ácido málico', 'Cenizas', 'Alcal. cenizas',
             'Magnesio', 'Fenoles tot.', 'Flavonoides', 'Fenoles no flav.',
             'Proantocian.', 'Intensidad', 'Tono', 'OD280/315', 'Prolina']
ax.set(yticks=[0, 1], yticklabels=['CP1', 'CP2'],
       xticks=np.arange(len(etiquetas)),
       xticklabels=etiquetas, title='Cargas de las dos componentes')
ax.tick_params(axis='x', labelrotation=65, labelsize=7)
fig.colorbar(im, ax=ax, label='Coeficiente')
fig.tight_layout()
fig.savefig(OUT / 'cargas_pca.png', bbox_inches='tight')
plt.show()

C:\Users\karlo\AppData\Local\Temp\ipykernel_29204\3124952587.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\karlo\AppData\Local\Temp\ipykernel_29204\3124952587.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
fig, axs = plt.subplots(1, 2, figsize=(9, 3.4))
axs[0].bar(range(1, len(singular)+1), singular**2 / np.sum(singular**2),
           color='#335f72')
axs[0].set(xlabel='Componente', ylabel='Fracción de varianza',
           title='Espectro SVD')
dendrogram(linkage(X, method='ward'), truncate_mode='lastp', p=10,
           show_leaf_counts=True, ax=axs[1], color_threshold=0)
axs[1].set(xlabel='Ramas finales', ylabel='Distancia', title='Fusiones de Ward')
fig.tight_layout()
fig.savefig(OUT / 'svd_ward.png', bbox_inches='tight')
plt.show()

resumen = {'muestras': len(X), 'variables': X.shape[1],
           'silueta_k3': float(comparacion.loc[0, 'silueta']),
           'silueta_ward': float(comparacion.loc[1, 'silueta']),
           'silueta_gmm': float(comparacion.loc[2, 'silueta']),
           'ari_k3': float(comparacion.loc[0, 'ARI externo']),
           'ari_ward': float(comparacion.loc[1, 'ARI externo']),
           'ari_gmm': float(comparacion.loc[2, 'ARI externo']),
           'varianza_pca2': varianza_2d, 'error_svd2': float(error_relativo),
           'diferencia_pca_svd': float(coincidencia),
           'mejor_k_silueta': int(sweep.loc[sweep.silueta.idxmax(), 'k']),
           'silueta_mejor': float(sweep.silueta.max())}
(OUT / 'resumen.json').write_text(json.dumps(resumen, indent=2), encoding='utf-8')
display(pd.Series(resumen, name='Resultado'))

C:\Users\karlo\AppData\Local\Temp\ipykernel_29204\599994614.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


muestras              1.780000e+02
variables             1.300000e+01
silueta_k3            2.848589e-01
silueta_ward          2.774440e-01
silueta_gmm           2.844213e-01
ari_k3                8.974950e-01
ari_ward              7.899332e-01
ari_gmm               8.803998e-01
varianza_pca2         5.540634e-01
error_svd2            6.677849e-01
diferencia_pca_svd    3.330669e-15
mejor_k_silueta       3.000000e+00
silueta_mejor         2.848589e-01
Name: Resultado, dtype: float64

## Alcance

La mezcla gaussiana y K-means hacen supuestos diferentes; una mejor silueta no prueba que se hayan recuperado las clases de procedencia. Una figura de dos componentes pierde parte de la información original. La conclusión se limita a este conjunto, esta escala y estos parámetros.